# Base path

In [0]:
# Define base path containing weather data
base_path = "/Volumes/weather_catalog/weather_platform/weather_data"

# Imports

In [0]:
# Import PySpark functions for data filtering, aggregations, and analytics
from pyspark.sql.functions import col, avg, sum, round, max, min

# Loading Silver and Gold path

In [0]:
# Define paths to Silver (cleaned data) and Gold (aggregated data) layers
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

# Loading Silver and Gold dfs

In [0]:
# Load Silver layer and all 5 Gold layer aggregations for analytics
# Gold tables: yearly_city_summary, monthly_city_summary, rainfall_ranking, temperature_trend, weather_extremes
silver_df = spark.read.parquet(silver_path)

yearly_df = spark.read.parquet(f"{gold_path}/yearly_city_summary")
monthly_df = spark.read.parquet(f"{gold_path}/monthly_city_summary")
rainfall_df = spark.read.parquet(f"{gold_path}/rainfall_ranking")
trend_df = spark.read.parquet(f"{gold_path}/temperature_trend")
extremes_df = spark.read.parquet(f"{gold_path}/weather_extremes")

print("Datasets loaded successfully")
print(f"Silver rows: {silver_df.count()}")

# Creating widgets

In [0]:
# Create interactive widgets for user-driven analysis
# City dropdown: select one of 10 Indian cities
# Date range: filter data between start_date and end_date
dbutils.widgets.dropdown(
    "city",
    "chennai",
    [
        "chennai",
        "coimbatore",
        "mumbai",
        "delhi",
        "bangalore",
        "hyderabad",
        "kolkata",
        "pune",
        "ahmedabad",
        "kochi"
    ]
)

dbutils.widgets.text("start_date", "2020-01-01")
dbutils.widgets.text("end_date", "2024-12-31")

print("Widgets created successfully")

# Read widget values

In [0]:
# Retrieve user-selected values from widgets for filtering
selected_city = dbutils.widgets.get("city")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print(f"City       : {selected_city}")
print(f"Start Date : {start_date}")
print(f"End Date   : {end_date}")

# Filter data based on user's selection

In [0]:
# Filter Silver data based on user selections
# This dataset is used for widget-driven analytics throughout the notebook
filtered_df = (
    silver_df
    .filter(col("city") == selected_city)
    .filter(col("date").between(start_date, end_date))
)

print(f"Filtered Rows: {filtered_df.count()}")

display(filtered_df)

# 1. Average Teamprature in the selected period

In [0]:
# Analytics 1: Calculate summary statistics for selected city and date range
# Shows average temperature and total precipitation for the filtered period
selected_period_summary = (
    filtered_df
    .groupBy("city")
    .agg(
        round(avg("temperature_c"), 2).alias("avg_temperature_c"),
        round(sum("precipitation_mm"), 2).alias("total_precipitation_mm")
    )
)

display(selected_period_summary)

# 2. Average temperature per city per year

In [0]:
# Analytics 2: Display yearly trends for all cities
# Shows average temperature and total precipitation per city per year
display(
    yearly_df
    .select(
        "year",
        "city",
        "avg_temperature_c",
        "total_precipitation_mm"
    )
    .orderBy("year", "city")
)

Databricks visualization. Run in Databricks to view.

# 3. Highest Rainfall City in a Given Period

In [0]:
# Analytics 3a: Calculate total rainfall for the selected city in the date range
# Returns only one row since filtered_df contains data for the selected city only
rainfall_period = (
    filtered_df
    .groupBy("city")
    .agg(
        round(
            sum("precipitation_mm"), 2).alias("total_precipitation_mm")
    )
    .orderBy(
        col("total_precipitation_mm").desc()
    )
)

display(rainfall_period)

In [0]:
# Analytics 3b: Rank all cities by total rainfall in the selected date range
# Unlike 3a, this queries all cities (not just selected_city) for comparison
rainfall_ranking_period = (
    silver_df
    .filter(col("date").between(start_date, end_date))
    .groupBy("city")
    .agg(
        round(
            sum("precipitation_mm"),2).alias("total_precipitation_mm")
    )
    .orderBy(
        col("total_precipitation_mm").desc()
    )
)

display(rainfall_ranking_period)

Databricks visualization. Run in Databricks to view.

# 4. Temperature Variation Across Years for a City

In [0]:
# Analytics 4: Show year-over-year temperature change for selected city
# Uses pre-calculated temperature_trend from Gold layer (includes lag calculations)
selected_city_trend = (
    trend_df
    .filter(col("city") == selected_city)
    .select(
        "year",
        "city",
        "avg_temperature_c",
        "temperature_change"
    )
    .orderBy("year")
)

display(selected_city_trend)

Databricks visualization. Run in Databricks to view.

# 5. Simple Weather Trend Analysis Over Time

In [0]:
# Analytics 5: Display monthly weather patterns for selected city
# Shows temperature and rainfall trends at monthly granularity
monthly_city_trend = (
    monthly_df
    .filter(col("city") == selected_city)
    .orderBy("year", "month")
)

display(monthly_city_trend)

Databricks visualization. Run in Databricks to view.

# 6. Weather Extremes analysis

In [0]:
# Analytics 6: Display weather extremes for selected city across all years
# Shows highest/lowest temperatures and maximum daily rainfall per year
selected_city_extremes = (
    extremes_df
    .filter(col("city") == selected_city)
    .orderBy("year")
)

display(selected_city_extremes)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

# 7. Overall hottest City-Year

In [0]:
# Analytics 7: Find top 5 hottest city-year combinations across entire dataset
display(yearly_df.orderBy(col("avg_temperature_c").desc()).limit(5))

# 8. Overall Coldest City-Year

In [0]:
# Analytics 8: Find top 5 coldest city-year combinations across entire dataset
display(yearly_df.orderBy(col("avg_temperature_c")).limit(5))

# 9. Overall Rainiest City-Year

In [0]:
# Analytics 9: Find top 5 rainiest city-year combinations across entire dataset
display(yearly_df.orderBy(col("total_precipitation_mm").desc()).limit(5))

# 10. Overall Driest City-Year

In [0]:
# Analytics 10: Find top 5 driest city-year combinations across entire dataset
display(yearly_df.orderBy(col("total_precipitation_mm")).limit(5))

# 11. Temperature vs Rainfall Relationship

In [0]:
# Analytics 11: Explore temperature vs rainfall relationship across all cities and years
# Displays full yearly_city_summary for correlation analysis
display(yearly_df)

Databricks visualization. Run in Databricks to view.

# 12. Climate Profile - Selected City

In [0]:
# Analytics 12: Create comprehensive climate profile for selected city and date range
# Includes averages, extremes, and total rainfall - useful for city climate characterization
city_climate_profile = (
    filtered_df
    .groupBy("city")
    .agg(
        round(avg("temperature_c"), 2).alias("average_temperature_c"),
        round(sum("precipitation_mm"), 2).alias("total_rainfall_mm"),
        round(max("temperature_c"), 2).alias("highest_temperature_c"),
        round(min("temperature_c"), 2).alias("lowest_temperature_c"),
        round(max("precipitation_mm"), 2).alias("highest_daily_rainfall_mm")
    )
)

display(city_climate_profile)

# 13. Warmest Year Analysis

In [0]:
# Analytics 13: Identify warmest years by averaging temperatures across all 10 cities
# Shows year-by-year warming/cooling trends at national level
warmest_year_df = (
    yearly_df
    .groupBy("year")
    .agg(
        round(
            avg("avg_temperature_c"),2).alias("average_temperature_all_cities")
    )
    .orderBy("year")
)

display(warmest_year_df)

Databricks visualization. Run in Databricks to view.

# 14. Rainfall Distribution by city

In [0]:
# Analytics 14: Compare total rainfall distribution across all 10 cities
# Useful for identifying wettest and driest regions overall
rainfall_share_df = (
    yearly_df
    .groupBy("city")
    .agg(
        round(
            sum("total_precipitation_mm"),
            2
        ).alias("total_rainfall_mm")
    )
    .orderBy(col("total_rainfall_mm").desc())
)

display(rainfall_share_df)

Databricks visualization. Run in Databricks to view.

# 15. Source Comparsion 

In [0]:
# Analytics 15: Compare data quality between Open-Meteo and NASA POWER sources
# Validates consistency of temperature and precipitation measurements across sources
source_comparison = (
    silver_df
    .groupBy("source")
    .agg(
        round(avg("temperature_c"), 2).alias("avg_temperature_c"),
        round(avg("precipitation_mm"), 2).alias("avg_precipitation_mm")
    )
)

display(source_comparison)